# Exp02C — τ=0 matched no-residual-dropout control

**Purpose:** Add only `--pdrop-res 0.0` to the Exp02 controlled retraining sweep. Keep RVQ's independent `--rvq-quantize-dropout-prob 0.2` unchanged. Reuse the established seeded compatibility launcher, 300k training parameters, best-validation-FID selection and Exp02B6 20-repeat test evaluator. Do not touch original Exp02 checkpoints.

Run sequentially using the **Python 3** kernel. The training launch cell starts one detached process; the evaluation cell must be run after training has finished. Public, output-free reproduction notebook. The matched τ=0 condition has been completed and reported in the 2026 THMS revision. Do not re-launch training if the verified checkpoint is already present.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, shlex, subprocess, time, re, signal
import numpy as np, pandas as pd
PGR=Path('.'); PY=Path('venv/bin/python')
PRE=PGR/'revision_preflight'; WRAPPER=PRE/'train_pg_tokenizer_exp02_seeded_compat.py'
ORIG=PGR/'revision_results/exp02_tau'; TRAIN=PGR/'exp_result_revision_tau_sensitivity'
OUT=PGR/'exp_result_revision_tau0_matched'; LOG=PGR/'revision_logs/exp02c_tau0'; RES=PGR/'revision_results/exp02c_tau0'
EXP='exp_pg_tokenizer_tau_0p0_seed123'; TAU=0.0
for path in (PGR,PY,WRAPPER,ORIG/'exp02b_test_runs.csv',ORIG/'exp02b_test_summary.csv',ORIG/'exp02_validation_audit.csv'):
    assert path.exists(), f'Missing prerequisite: {path}'
for d in (OUT,LOG,RES):d.mkdir(parents=True,exist_ok=True)
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
print('Interpreter:',PY,'\nTrain wrapper:',WRAPPER,'\nOutput:',OUT)


In [ ]:
# Recover exact COMMON_ARGS from the existing Exp02 configuration, rather than retyping parameters.
CFG=PRE/'exp02_tau_queue_config.json'
assert CFG.exists(), 'Run Exp02 first to create its frozen command configuration.'
cfg=json.loads(CFG.read_text()); run01=next(r for r in cfg['runs'] if abs(float(r['tau'])-0.1)<1e-9)
old=list(run01['command'])
assert '--pdrop-res' in old and '--exp-name' in old and '--out-dir' in old
cmd=old.copy()
def setflag(flag,value):
    ix=cmd.index(flag);cmd[ix+1]=str(value)
setflag('--pdrop-res','0.0');setflag('--out-dir',OUT);setflag('--exp-name',EXP)
assert cmd[0]==str(PY) and Path(cmd[2]).resolve()==WRAPPER.resolve()
assert '--rvq-quantize-dropout-prob' in cmd and cmd[cmd.index('--rvq-quantize-dropout-prob')+1]=='0.2'
assert cmd[cmd.index('--seed')+1]=='123' and cmd[cmd.index('--total-iter')+1]=='300000'
assert cmd[cmd.index('--batch-size')+1]=='256'
assert cmd[cmd.index('--lr')+1]=='2e-4'
origcheck=old.copy()
for flag in ('--pdrop-res','--out-dir','--exp-name'):
    origcheck[origcheck.index(flag)+1]=cmd[cmd.index(flag)+1]
assert origcheck==cmd, 'Only three fields must differ from Exp02 tau=0.1'
assert 'random.seed(123)' in WRAPPER.read_text() or '_exp02_python_random.seed(123)' in WRAPPER.read_text()
manifest={'experiment':'Exp02C tau=0.0 matched control','baseline_exp02_command':old,'command':cmd,'seeded_wrapper_sha256':sha(WRAPPER),'loss_cfg_sha256':sha(PGR/'configs/exp_dec/losses.yaml'),'original_exp02_config_sha256':sha(CFG)}
MAN=RES/'exp02c_train_manifest.json'
if MAN.exists():assert json.loads(MAN.read_text())['command']==cmd, 'Existing manifest differs'
else: MAN.write_text(json.dumps(manifest,indent=2))
print('Only changed flags: tau, out-dir, exp-name')
print(shlex.join(cmd))


## Launch τ=0 training (300,000 iterations)
The launcher uses a private output directory and refuses to overwrite an existing checkpoint. It records PID/log; do not run twice while active.

In [ ]:
TRAIN_LOG=LOG/'exp02c_tau0_training.log';PID=LOG/'exp02c_tau0.pid'
existing=list((OUT/EXP).rglob('net_best_fid.pth')) if (OUT/EXP).exists() else []
assert not existing, f'Checkpoint already exists; do not restart: {existing}'
if PID.exists():
    p=int(PID.read_text()); os.kill(p,0)
    raise RuntimeError(f'Existing training process is running: PID={p}')
assert not TRAIN_LOG.exists(), f'Existing training log found: {TRAIN_LOG}; inspect before restart'
f=open(TRAIN_LOG,'w')
proc=subprocess.Popen(cmd,cwd=PGR,stdout=f,stderr=subprocess.STDOUT,start_new_session=True)
f.close();PID.write_text(str(proc.pid))
print('Launched PID:',proc.pid,'\nLog:',TRAIN_LOG)


In [ ]:
# Monitor: re-run this cell safely.
if TRAIN_LOG.exists():
    lines=TRAIN_LOG.read_text(errors='replace').splitlines()
    print('\n'.join(lines[-30:]))
if PID.exists():
    pid=int(PID.read_text())
    try:os.kill(pid,0);print('PID exists:',pid,'(may be running or zombie; confirm via ps)')
    except ProcessLookupError:print('PID no longer running:',pid)


## Validation, best-FID checkpoint and pose-codebook orthogonality
Discover the official best-validation-FID checkpoint first. Verify its SHA256 and the `nb_iter` field (39,000) before reporting the paper's τ=0 result. Compute the identical squared, off-diagonal cosine-similarity diagnostic used in Exp02; the expected reported value is **0.109110** for the `(392, 512)` pose codebook.

**Security:** load only the authors' trusted checkpoint: the serialized `.pth` file may contain Python objects. No checkpoint is included in this handoff.

In [ ]:
from pathlib import Path
import subprocess

print("Experiment:", EXP)
print("Output:", OUT)

# 1. 학습 프로세스 확인
if PID.exists():
    pid = PID.read_text().strip()
    print("\nTraining PID:", pid)
    subprocess.run(
        ["ps", "-p", pid, "-o", "pid,stat,etime,%cpu,%mem,args"],
        check=False
    )
else:
    print("\nPID file not found")

# 2. 학습 로그 확인
if TRAIN_LOG.exists():
    lines = TRAIN_LOG.read_text(errors="replace").splitlines()
    print("\n=== Last 40 training log lines ===")
    print("\n".join(lines[-40:]))
else:
    print("\nTraining log not found")

# 3. 체크포인트 전체 검색
print("\n=== Checkpoints ===")
checkpoints = list(OUT.rglob("*.pth"))
for p in checkpoints[:30]:
    print(p, f"({p.stat().st_size / 1024**2:.1f} MB)")

print(f"\nTotal checkpoints: {len(checkpoints)}")

In [ ]:
candidates=list((OUT/EXP).rglob('net_best_fid.pth'))
assert len(candidates)==1, f'Expected exactly one best-FID checkpoint, got: {candidates}'
CKPT=candidates[0];CKPT_DIR=CKPT.parent
assert (CKPT_DIR/'arguments.yaml').exists(), 'Evaluation requires arguments.yaml beside checkpoint'
text=TRAIN_LOG.read_text(errors='replace')
assert '300000' in text or re.search(r'Train\. Iter\s+299999\s*:',text), 'Training completion not yet confirmed from log; inspect manually'
print('Selected checkpoint:',CKPT,'\nSHA256:',sha(CKPT))
print('Log end:', '\n'.join(text.splitlines()[-12:]))


### Exp02C orthogonality and checkpoint authenticity audit
This is the same codebook metric used by the original Exp02 post-run validation audit. It requires the official, author-owned checkpoint and writes only a small JSON record, not model weights.

In [ ]:
import torch

EXPECTED_TAU0_SHA256 = 'c1761ac08c32a8ae249118d6827eb68e461940029822ad17f102de6497df6ad8'
EXPECTED_TAU0_ITER = 39000
EXPECTED_TAU0_ORTH = 0.109110  # Rounded paper/reviewer result
actual_sha = sha(CKPT)
assert actual_sha == EXPECTED_TAU0_SHA256, (
    f'Checkpoint SHA256 mismatch; this is not the reported τ=0 checkpoint: {actual_sha}'
)
# Only load the authors' trusted checkpoint (torch.load can deserialize Python objects).
obj = torch.load(CKPT, map_location='cpu', weights_only=False)
state = obj.get('net', obj)
keys = [k for k in state if k.endswith('pc_encoder.codebook.weight')]
if not keys:
    keys = [k for k in state if 'pc_encoder' in k and k.endswith('codebook.weight')]
assert len(keys) == 1, f'Expected one pose codebook, got: {keys}'
w = state[keys[0]].float()
assert tuple(w.shape) == (392, 512), f'Unexpected codebook shape: {w.shape}'
w_norm = w / (w.norm(dim=1, keepdim=True) + 1e-12)
gram = w_norm @ w_norm.T
mask = ~torch.eye(w.shape[0], dtype=torch.bool)
orth_tau0 = float(gram[mask].square().mean().item())
ckpt_iteration = obj.get('nb_iter')
assert int(ckpt_iteration) == EXPECTED_TAU0_ITER, ckpt_iteration
assert abs(orth_tau0 - EXPECTED_TAU0_ORTH) < 0.000005, orth_tau0
orth_audit = {
    'experiment': 'Exp02C', 'tau': 0.0,
    'checkpoint_file': CKPT.name, 'checkpoint_sha256': actual_sha,
    'best_validation_fid_iteration': int(ckpt_iteration),
    'codebook_parameter': keys[0], 'codebook_shape': list(w.shape),
    'orthogonality': orth_tau0,
    'orthogonality_definition': 'mean squared off-diagonal cosine similarity of L2-normalized pose-codebook vectors',
    'checkpoint_not_redistributed': True,
}
(RES/'exp02c_tau0_orthogonality_audit.json').write_text(
    json.dumps(orth_audit, indent=2) + '\n', encoding='utf-8'
)
print('Best FID checkpoint iteration:', ckpt_iteration)
print('SHA256:', actual_sha)
print('Pose-codebook shape:', tuple(w.shape))
print('Orthogonality:', f'{orth_tau0:.6f}')

## 20-repeat HumanML3D test evaluation
Reuse the existing Exp02B6 compatibility evaluator (generated when Exp02B6 was run). Keep source and argument provenance identical. Test outputs go to a separate directory.

In [ ]:
EVAL_COMPAT=PRE/'eval_pg_tokenizer_exp02b_compat.py'
assert EVAL_COMPAT.exists(), 'Run Exp02B6 compatibility setup cell before this evaluation.'
assert 'repeat_time = 20' in (PGR/'eval_pg_tokenizer.py').read_text(), 'Expected 20-repeat evaluator source'
EVAL_OUT=PGR/'eval_output_revision_tau0_matched';EVAL_OUT.mkdir(exist_ok=True)
EVAL_CMD=[str(PY),'-u',str(EVAL_COMPAT),'--batch-size','256','--lr','2e-4','--total-iter','300000','--lr-scheduler','200000','--nb-code','392','--down-t','2','--depth','3','--dilation-growth-rate','3','--out-dir',str(EVAL_OUT),'--dataname','t2m','--vq-act','relu','--exp-name','tau_0p0','--use-keywords','--dec-checkpoint-folder',str(CKPT_DIR),'--output-emb-width','392','--num_workers','4','--seed','123']
EVAL_LOG=LOG/'exp02c_tau0_test_20repeat.log'
print(shlex.join(EVAL_CMD))
if EVAL_LOG.exists() and 'final result:' in EVAL_LOG.read_text(errors='replace').lower():
    print('Completed log exists; evaluation skipped')
else:
    assert not EVAL_LOG.exists(), f'Partial log exists: {EVAL_LOG}; inspect before explicitly deleting or moving'
    with EVAL_LOG.open('w') as f: p=subprocess.run(EVAL_CMD,cwd=PGR,stdout=f,stderr=subprocess.STDOUT)
    assert p.returncode==0 and 'final result:' in EVAL_LOG.read_text(errors='replace').lower(),f'Evaluation failed: {EVAL_LOG}'
print('Evaluation log:',EVAL_LOG)


In [ ]:
# Parse 20 uniquely timestamped evaluation entries using Exp02B6's exact logging format.
pat=re.compile(r'^\d{4}-\d{2}-\d{2} .*?Eva\. Iter 0 :, FID\. (?P<fid>[-+0-9.eE]+), Diversity Real\. (?P<div_real>[-+0-9.eE]+), Diversity\. (?P<div>[-+0-9.eE]+), R_precision_real\. \[[^\]]+\], R_precision\. \[(?P<rp>[^\]]+)\], matching_score_real\. (?P<match_real>[-+0-9.eE]+), matching_score_pred\. (?P<matching>[-+0-9.eE]+), MPJPE:(?P<mpjpe>[-+0-9.eE]+), PA-MPJPE:(?P<pampjpe>[-+0-9.eE]+), ACCEL:(?P<accel>[-+0-9.eE]+)')
rows=[]
for line in EVAL_LOG.read_text(errors='replace').splitlines():
    m=pat.search(line)
    if not m:continue
    g=m.groupdict();r=[float(x) for x in g['rp'].split()]
    assert len(r)==3
    rows.append(dict(tau=0.0,repeat=len(rows)+1,fid=float(g['fid']),diversity=float(g['div']),top1=r[0],top2=r[1],top3=r[2],matching=float(g['matching']),mpjpe=float(g['mpjpe']),pampjpe=float(g['pampjpe']),accel=float(g['accel'])))
assert len(rows)==20,f'Expected 20 timestamped evaluations; found {len(rows)}'
newruns=pd.DataFrame(rows);newruns.to_csv(RES/'exp02c_tau0_test_runs.csv',index=False)
def ci(x):return 1.96*np.std(np.asarray(x,dtype=float),ddof=0)/np.sqrt(len(x))
newsum={'tau':0.0,'n':20,'best_val_iter':int(ckpt_iteration), 'orthogonality':orth_tau0, 'checkpoint':str(CKPT),'checkpoint_sha256':sha(CKPT)}
for m in ['fid','diversity','top1','top2','top3','matching','mpjpe','pampjpe','accel']:
    newsum[f'{m}_mean']=float(newruns[m].mean());newsum[f'{m}_ci95']=float(ci(newruns[m]))
pd.DataFrame([newsum]).to_csv(RES/'exp02c_tau0_test_summary.csv',index=False)
print(pd.DataFrame([newsum]).T)


## Five-condition integration (without modifying Exp02 originals)
Combine the independently trained τ=0 condition with the four original Exp02/Exp02B test conditions into **new** five-condition files. The τ=0 codebook orthogonality is measured by the SHA256-verified audit cell above; the four existing orthogonality scores come from the original Exp02 validation audit. Do not combine legacy Table III and the controlled Table V runs.

In [ ]:
original=pd.read_csv(ORIG/'exp02b_test_runs.csv')
assert all((original.groupby('tau').size()==20)), 'Original Exp02 results are incomplete'
combined=pd.concat([newruns,original],ignore_index=True).sort_values(['tau','repeat'])
assert set(np.round(combined.tau.unique(),1))=={0.0,0.1,0.3,0.7,0.9}
combined.to_csv(RES/'exp02c_five_tau_test_runs.csv',index=False)
summary=[]
for tau,group in combined.groupby('tau'):
    row={'tau':tau,'n':len(group)}
    for m in ['fid','top1','top2','top3','matching','mpjpe','pampjpe','accel']:
        row[m+'_mean']=group[m].mean();row[m+'_ci95']=ci(group[m])
    summary.append(row)
summ=pd.DataFrame(summary)
original_summary=pd.read_csv(ORIG/'exp02b_test_summary.csv')
if 'orthogonality' in original_summary.columns:
    for _,r in original_summary.iterrows():
        summ.loc[np.isclose(summ.tau,r.tau),'orthogonality']=r.orthogonality
assert 'orth_audit' in globals(), 'Run the checkpoint orthogonality audit cell first'
summ.loc[np.isclose(summ.tau,0.0),'orthogonality']=orth_audit['orthogonality']
assert summ['orthogonality'].notna().all(), 'Missing orthogonality in five-condition summary'
summ.to_csv(RES/'exp02c_five_tau_summary.csv',index=False)
print('Merged five conditions with verified τ=0 orthogonality')
display(summ)


### Interpretation guardrails
- `τ=0` here disables **residual dropout**, not RVQ quantizer dropout.
- Do not merge legacy Table III values into this matched-control table.
- 20 test repeats estimate **evaluation variability** for one trained seed, not across-training-seed uncertainty.
- Check the source of legacy Table III/Table II results independently before claiming seed alone explains their differences.
- If the original Exp02 wrapper, loss YAML, evaluator, or data changed after training, compare frozen hashes before asserting exact parity.
- Do not report a completed τ=0 result until training, best-checkpoint audit, and 20-repeat test all pass.